In [30]:
from torch.utils.data import random_split
import torchvision.models.segmentation as segmentation
import torch.nn as nn
import torch.optim as optim
import numpy as np


In [31]:
import pandas as pd
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet

try: # disable certificate verification, needed on MacOS
    import ssl
    ssl._create_default_https_context = ssl._create_unverified_context
except ImportError:
    pass  # SSL module not available, skipping workaround

class SegmentationTransform:
    def __init__(self, size=(256, 256)):
        self.size = size
        self.image_transform = transforms.Compose([
            transforms.Resize(self.size),
            transforms.ToTensor()
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize(self.size, interpolation=transforms.InterpolationMode.NEAREST),
            transforms.PILToTensor()
        ])

    def __call__(self, image, target):
        image = self.image_transform(image)
        target = self.mask_transform(target).squeeze(0).long() - 1  # [1, H, W] → [H, W], label remap
        return image, target

train_dataset = OxfordIIITPet(
    root='.',
    target_types='segmentation',
    download=True,
    transforms=SegmentationTransform(size=(256, 256))
)

batch_size = 32  # TODO: change to your needs

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

def load_images(filename: str, image_shape: tuple = (3, 256, 256)):
    df = pd.read_csv(filename)

    images = []
    for _, row in df.iterrows():
        flat = eval(row['image'])  # Convert stringified list back to list
        tensor = torch.tensor(flat, dtype=torch.float32).reshape(image_shape)
        images.append(tensor)

    return torch.stack(images)  # [B, C, H, W]



100%|██████████| 792M/792M [00:53<00:00, 14.8MB/s] 
100%|██████████| 19.2M/19.2M [00:01<00:00, 10.8MB/s]


We split our training data to evaluate the model and implement early stopping.

In [4]:
# Dataset and split.
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

# Create loaders
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)



We are using DeepLabV3 with a ResNet-50 backbone (deeplabv3_resnet50) pretrained model from torchvision.models

In [6]:
class SegmentationModel(nn.Module):
    def __init__(self, num_classes=3, freeze=True):
        super().__init__()
        # Load pretrained DeepLabV3
        #weights = segmentation.DeepLabV3_ResNet50_Weights.DEFAULT
        #self.model = segmentation.deeplabv3_resnet50(weights=weights)
        # Load the lightweight MobileNetV3 DeepLab model (because DeepLabV3 slow af - 2 epochs after 3 hours, this one took only 46 minutes)
        weights = segmentation.DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT
        self.model = segmentation.deeplabv3_mobilenet_v3_large(weights=weights)

        # Freeze the backbone based on the parameter, as seen in NN_finetuning.ipynb
        if freeze:
            for param in self.model.backbone.parameters():
                param.requires_grad = False

        # Replace the classifier head for 3 classes (background, pet, edge)
        in_channels = self.model.classifier[4].in_channels
        self.model.classifier[4] = nn.Conv2d(in_channels, num_classes, kernel_size=1, stride=1)

        # Handle auxiliary classifier if present
        if self.model.aux_classifier is not None:
            in_channels_aux = self.model.aux_classifier[4].in_channels
            self.model.aux_classifier[4] = nn.Conv2d(in_channels_aux, num_classes, kernel_size=1, stride=1)

    def forward(self, x):
        # torchvision segmentation models output a dictionary. We want the main output.
        return self.model(x)['out']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SegmentationModel(freeze=True).to(device)

Training and evaluation loop. We take our model and feed it data to build best model(takes a long time).

In [7]:
# Your provided IoU function
def compute_iou(mask_true: np.ndarray, mask_pred: np.ndarray, num_classes: int = 3) -> float:
    """Computes mean Intersection over Union (mIoU) for multi-class masks."""
    ious = []

    for cls in range(num_classes):
        true_cls = mask_true == cls
        pred_cls = mask_pred == cls

        intersection = np.logical_and(true_cls, pred_cls).sum()
        union = np.logical_or(true_cls, pred_cls).sum()

        if union == 0:
            continue  # skip this class — not present in either pred or true

        iou = intersection / union
        ious.append(iou)

    if not ious:
        return 1.0  # If no classes are present in either mask, define mIoU as perfect

    return np.mean(ious)
#calculating section over union

def train(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        avg_loss = total_loss / len(dataloader)
    print(f"Train Loss: {avg_loss:.4f}")
    return total_loss / len(dataloader)
#use the same code as in practicals but keep the result


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    val_ious = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item()

            # Calculate metrics
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            masks_np = targets.cpu().numpy()
            for i in range(preds.shape[0]):
                val_ious.append(compute_iou(masks_np[i], preds[i]))

    avg_loss = total_loss / len(dataloader)
    avg_iou = np.mean(val_ious)
    return avg_loss, avg_iou
#adapt code from practicals for semantic segmentation - classifying every single pixel in the image

# Setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
epochs = 10
best_iou = 0.0
# Execution with Early Stopping
for epoch in range(epochs):
    train_loss = train(model, train_loader, criterion, optimizer, device)
    val_loss, val_iou = validate(model, val_loader, criterion, device)

    print(f"Epoch {epoch + 1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val mIoU: {val_iou:.4f}")

    # Simple Early Stopping mechanism (saving the best model)
    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), 'best_model.pth')
        print("Best model saved!")

Train Loss: 0.3152
Epoch 1/10 - Train Loss: 0.3152, Val Loss: 0.2831, Val mIoU: 0.6832
Best model saved!
Train Loss: 0.2565
Epoch 2/10 - Train Loss: 0.2565, Val Loss: 0.2692, Val mIoU: 0.7017
Best model saved!
Train Loss: 0.2403
Epoch 3/10 - Train Loss: 0.2403, Val Loss: 0.2596, Val mIoU: 0.7027
Best model saved!
Train Loss: 0.2313
Epoch 4/10 - Train Loss: 0.2313, Val Loss: 0.2493, Val mIoU: 0.7117
Best model saved!
Train Loss: 0.2192
Epoch 5/10 - Train Loss: 0.2192, Val Loss: 0.2614, Val mIoU: 0.7134
Best model saved!
Train Loss: 0.2076
Epoch 6/10 - Train Loss: 0.2076, Val Loss: 0.2560, Val mIoU: 0.7150
Best model saved!
Train Loss: 0.1997
Epoch 7/10 - Train Loss: 0.1997, Val Loss: 0.2566, Val mIoU: 0.7111
Train Loss: 0.1899
Epoch 8/10 - Train Loss: 0.1899, Val Loss: 0.2714, Val mIoU: 0.7079
Train Loss: 0.1865
Epoch 9/10 - Train Loss: 0.1865, Val Loss: 0.2773, Val mIoU: 0.7055
Train Loss: 0.1818
Epoch 10/10 - Train Loss: 0.1818, Val Loss: 0.2693, Val mIoU: 0.7114


In [10]:
# 1. Load test data and best model
test_tensor = load_images('test_images.csv').to(device)
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# 2. Predict
predictions = []
with torch.no_grad():
    outputs = model(test_tensor)
    # Shape becomes [Batch, 256, 256]
    preds = torch.argmax(outputs, dim=1).cpu().numpy()

    # 3. Flatten and stringify
    for i in range(preds.shape[0]):
        flat_mask = preds[i].flatten().tolist()
        predictions.append(str(flat_mask))

# 4. Save to CSV
submission_df = pd.DataFrame({
    'id': range(1, len(predictions) + 1),
    'mask': predictions,
    'Usage': 'Public'
})
submission_df.to_csv('submission.csv', index=False)
#print("Submission saved!")